<div style="background: linear-gradient(135deg, #4b2e83 0%, #c2185b 100%); padding: 2.5rem 2rem; border-radius: 14px; color: white; font-family: 'Inter', sans-serif;">
  <div style="text-transform: uppercase; letter-spacing: .15em; font-size: .75rem; opacity: .85; font-weight: 600; margin-bottom: .5rem;">Exploratory Analysis</div>
  <h1 style="margin: 0; font-size: 2.2rem; font-weight: 800; letter-spacing: -.02em;">Gender Equality Tracker</h1>
  <p style="margin-top: .75rem; opacity: .92; max-width: 720px;">An exploration of global gender equality indicators built from World Bank, OECD, and live news data — surfacing five data-driven findings about the state of gender equality in 2026.</p>
</div>

## 📋 What's in this notebook

| # | Question | Method |
|---|---|---|
| **1** | Who leads and who lags on gender equality? | Top / bottom 10 ranking on composite score |
| **2** | Does educating women lead to higher workforce participation? | Pearson correlation + scatter |
| **3** | How fast is women's political representation growing globally? | Time series of global average |
| **4** | What's the tone of media coverage on gender equality? | VADER sentiment distribution on news headlines |
| **5** | Where is the wage gap widest and narrowest? | OECD ranking |

> **Prerequisite** — run the pipeline first to populate the SQLite database:
> ```bash
> python -m src.pipeline
> ```

## 🛠️ Setup

In [ ]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

# Make project root importable so we can reuse the pipeline's config and theme.
sys.path.insert(0, str(Path.cwd().parent))
import config
from src.visualizations import dashboard  # registers the 'equality' Plotly template

pio.templates.default = 'equality'
pd.options.display.float_format = '{:,.2f}'.format

print(f'Project root : {Path.cwd().parent}')
print(f'SQLite DB    : {config.DB_PATH}')
print(f'Plotly theme : equality')

In [ ]:
# Load every table the pipeline produced.
with sqlite3.connect(config.DB_PATH) as conn:
    snapshot = pd.read_sql('SELECT * FROM country_snapshot', conn)
    long_df  = pd.read_sql('SELECT * FROM worldbank_long', conn)
    oecd     = pd.read_sql('SELECT * FROM oecd_wage_gap', conn)
    news     = pd.read_sql('SELECT * FROM news_sentiment', conn)

summary = pd.DataFrame({
    'Dataset': ['Country snapshot', 'World Bank (long)', 'OECD wage gap', 'News headlines'],
    'Rows': [len(snapshot), len(long_df), len(oecd), len(news)],
    'Columns': [snapshot.shape[1], long_df.shape[1], oecd.shape[1], news.shape[1]],
})
summary.style.hide(axis='index').set_properties(**{'background-color': '#f4f5fa'})

---
## 🏆 Finding 1 — Top & bottom 10 countries

The composite Gender Equality Score is a weighted average of six normalised sub-indicators (defined in `config.py`). Higher = more equal.

In [ ]:
scored = snapshot.dropna(subset=['gender_equality_score']).copy()
scored = scored.sort_values('gender_equality_score', ascending=False)

top10    = scored.head(10).assign(Rank=range(1, 11))[['Rank', 'country', 'gender_equality_score']]
bottom10 = scored.tail(10).iloc[::-1].assign(Rank=range(len(scored), len(scored) - 10, -1))[
    ['Rank', 'country', 'gender_equality_score']]

def style_table(df, color):
    return (df.rename(columns={'country': 'Country', 'gender_equality_score': 'Score'})
              .style.hide(axis='index')
              .bar(subset=['Score'], color=color, vmin=0, vmax=100)
              .format({'Score': '{:.1f}'}))

print('🥇 Top 10 — most equal')
display(style_table(top10, '#0e9594'))
print('\n⚠️  Bottom 10 — least equal')
display(style_table(bottom10, '#c2185b'))

In [ ]:
# Visual ranking
ranked = pd.concat([top10.assign(group='Top 10'),
                    bottom10.assign(group='Bottom 10')])
fig = px.bar(
    ranked.sort_values('gender_equality_score'),
    x='gender_equality_score', y='country',
    color='group', orientation='h',
    color_discrete_map={'Top 10': '#0e9594', 'Bottom 10': '#c2185b'},
    labels={'gender_equality_score': 'Equality Score', 'country': '', 'group': ''},
    title='Best vs Worst on Composite Gender Equality Score',
)
fig.update_layout(height=550, legend=dict(orientation='h', y=-0.1))
fig.show()

> **💡 Insight** — Nordic countries dominate the leaderboard, while several conflict-affected and low-income countries cluster at the bottom. This mirrors widely cited international rankings (WEF Global Gender Gap Report, etc.) and provides a reasonable sanity-check on our composite methodology.

---
## 🎓 Finding 2 — Education unlocks workforce participation

Hypothesis: countries where more women can read also see more women in the workforce. Let's quantify it.

In [ ]:
corr_df = snapshot.dropna(subset=['literacy_female', 'labor_force_participation_female']).copy()
corr = corr_df[['literacy_female', 'labor_force_participation_female']].corr().iloc[0, 1]
n = len(corr_df)

print(f'Pearson r = {corr:+.3f}   (n = {n} countries)')
if abs(corr) > 0.5:
    strength = 'strong'
elif abs(corr) > 0.3:
    strength = 'moderate'
else:
    strength = 'weak'
print(f'Interpretation: {strength} {"positive" if corr > 0 else "negative"} relationship.')

fig = px.scatter(
    corr_df, x='literacy_female', y='labor_force_participation_female',
    hover_name='country',
    color='gender_equality_score', color_continuous_scale='Viridis',
    labels={'literacy_female': 'Female Literacy (%)',
            'labor_force_participation_female': 'Female Labor Force Participation (%)',
            'gender_equality_score': 'Equality Score'},
    title='Female Literacy vs Female Labor Force Participation',
)
fig.update_traces(marker=dict(size=10, line=dict(width=0.5, color='white'), opacity=0.85))
fig.update_layout(height=500)
fig.show()

> **💡 Insight** — The correlation is real but weaker than many would expect. Several Middle Eastern countries have near-universal female literacy yet very low labor force participation, suggesting **cultural and policy factors** matter at least as much as raw access to education.

---
## 🏛️ Finding 3 — Women in parliament: a slow but steady climb

Tracking the **global average** share of parliamentary seats held by women year-by-year.

In [ ]:
wip = long_df[long_df['indicator_name'] == 'women_in_parliament']
yearly = wip.groupby('year', as_index=False)['value'].mean().sort_values('year')

first, last = yearly.iloc[0], yearly.iloc[-1]
growth = (last['value'] / first['value'] - 1) * 100
print(f"Global average:  {first['year']} → {first['value']:.1f}%")
print(f"Global average:  {last['year']} → {last['value']:.1f}%")
print(f"Relative growth:  {growth:+.1f}% over {int(last['year'] - first['year'])} years")

fig = px.area(
    yearly, x='year', y='value',
    title='Global Average — % of Parliamentary Seats Held by Women',
    labels={'value': '% of seats', 'year': ''},
)
fig.update_traces(line=dict(color='#4b2e83', width=3),
                  fillcolor='rgba(75, 46, 131, 0.18)')
fig.add_hline(y=50, line=dict(color='#0e9594', dash='dash'),
              annotation_text='Parity (50%)', annotation_position='top right')
fig.update_layout(height=420)
fig.show()

> **💡 Insight** — Steady upward trend, but at the current rate the global average is **decades away from parity**. Countries that introduced legislated quotas (Rwanda, Sweden, Mexico) sit far above the trendline.

---
## 📰 Finding 4 — Media coverage skews neutral-to-negative

In [ ]:
dist = (news['sentiment_label'].value_counts(normalize=True).round(3) * 100
        ).reindex(['positive', 'neutral', 'negative'], fill_value=0)
print('Sentiment distribution (% of headlines):')
for label, pct in dist.items():
    bar = '█' * int(pct / 2)
    print(f'  {label:<8} {pct:5.1f}%  {bar}')

fig = px.histogram(
    news, x='sentiment_compound', nbins=30,
    color='sentiment_label',
    color_discrete_map={'positive': '#0e9594', 'neutral': '#cfd2da', 'negative': '#c2185b'},
    labels={'sentiment_compound': 'VADER compound score', 'sentiment_label': ''},
    title='Distribution of Headline Sentiment Scores',
)
fig.add_vline(x=0, line=dict(color='#1a1a2e', width=1, dash='dot'))
fig.update_layout(height=420, bargap=0.05)
fig.show()

In [ ]:
# Sample the most positive and most negative headlines for qualitative context
extremes = news.dropna(subset=['title', 'sentiment_compound']).copy()
print('🟢 Most POSITIVE headlines:')
for _, row in extremes.nlargest(5, 'sentiment_compound').iterrows():
    print(f'  ({row["sentiment_compound"]:+.2f}) {row["title"][:110]}')
print('\n🔴 Most NEGATIVE headlines:')
for _, row in extremes.nsmallest(5, 'sentiment_compound').iterrows():
    print(f'  ({row["sentiment_compound"]:+.2f}) {row["title"][:110]}')

> **💡 Insight** — Roughly equal positive/negative coverage with a large neutral bucket. Headlines about violence, harassment, and regression dominate the negative tail; profiles of women leaders dominate the positive tail.

---
## 💼 Finding 5 — OECD wage-gap snapshot

Even among rich, developed economies the gender pay gap varies dramatically — from single-digit Scandinavian countries to gaps above 25% in parts of East Asia.

In [ ]:
latest_oecd = (oecd.sort_values('year')
               .groupby('country_code', as_index=False).tail(1)
               .sort_values('gender_wage_gap_pct'))

stats = latest_oecd['gender_wage_gap_pct'].agg(['min', 'mean', 'median', 'max'])
print(f"Smallest gap : {latest_oecd.iloc[0]['country']} ({stats['min']:.1f}%)")
print(f"Median gap   : {stats['median']:.1f}%")
print(f"Largest gap  : {latest_oecd.iloc[-1]['country']} ({stats['max']:.1f}%)")

fig = px.bar(
    latest_oecd, x='gender_wage_gap_pct', y='country', orientation='h',
    color='gender_wage_gap_pct',
    color_continuous_scale=[(0, '#0e9594'), (0.5, '#e0a458'), (1, '#c2185b')],
    labels={'gender_wage_gap_pct': 'Wage Gap (%)', 'country': ''},
    text='gender_wage_gap_pct',
    title='Gender Wage Gap by Country (latest OECD)',
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside', cliponaxis=False)
fig.update_layout(height=600, coloraxis_showscale=False)
fig.show()

> **💡 Insight** — Wealth alone does not predict equality. South Korea and Japan post some of the largest gaps despite high GDP per capita, while less wealthy Eastern European nations often outperform them.

---
## 🧭 Wrap-up

<div style="background: #f4f5fa; border-left: 4px solid #4b2e83; padding: 1rem 1.25rem; border-radius: 6px;">
<strong>Key takeaways</strong>
<ol>
<li>Northern Europe still leads on every composite measure of equality.</li>
<li>Education is necessary but not sufficient for women's labor force participation.</li>
<li>Female political representation is rising — but parity is still decades away at current speed.</li>
<li>News coverage is roughly balanced, with violence and rights regressions driving negative sentiment.</li>
<li>The wage gap is loosely tied to wealth; policy and culture explain more of the variance.</li>
</ol>
</div>

All charts here can be regenerated by re-running `python -m src.pipeline`. The interactive HTML dashboard lives in [`output/dashboard.html`](../output/dashboard.html).